# Colab bootstrap

Run the cell below once after connecting to a new Colab runtime. It mounts Google Drive, clones or updates the repository, installs the project with the annotation-app dependencies, and configures the shared runtime paths.

In [1]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/terriljoel/retrieval-grounded-remote-sensing.git"
REPO_REF = "feat/object-detection-YOLO"  # Change to main after merging.
REPO_DIR = Path("/content/retrieval-grounded-remote-sensing")
SHARED_ROOT = Path("/content/drive/Othercomputers/My laptop/shared_resources")

drive.mount("/content/drive")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Repository path exists but is not a Git clone: {REPO_DIR}")
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[annotation]"],
    check=True,
)

if not SHARED_ROOT.is_dir():
    raise FileNotFoundError(
        f"Shared resources were not found at {SHARED_ROOT}. "
        "Check the Google Drive computer and folder names."
    )

os.environ.update({
    "SHARED_RESOURCES_ROOT": str(SHARED_ROOT),
    "RAW_DATASET_ROOT": "/content/datasets/raw",
    "PROCESSED_DATASET_ROOT": "/content/datasets/processed",
    "MANIFEST_ROOT": str(SHARED_ROOT / "datasets" / "manifests"),
    "EXPERIMENT_OUTPUT_ROOT": str(SHARED_ROOT / "experiment_outputs"),
    "JOB_LOG_ROOT": str(SHARED_ROOT / "experiment_outputs" / "job_logs"),
    "INFERENCE_EXPORT_ROOT": str(SHARED_ROOT / "inference" / "object_detection_inference_new_remote_sensing_dataset_external-4"),
})

Path(os.environ["MANIFEST_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EXPERIMENT_OUTPUT_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["JOB_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

print(f"Ready. Working directory: {Path.cwd()}")
print("The detector commands and annotation app are ready.")

Mounted at /content/drive
Ready. Working directory: /content/retrieval-grounded-remote-sensing
The detector commands and annotation app are ready.


## Launch the annotation assistant (optional)

Run this cell only when you want to use the Streamlit app. It securely prompts for the NVIDIA API key when `NIM_API_KEY` is not already set, which also works when the Colab runtime is used through VS Code. The key remains only in the active runtime. The cell starts a temporary public Cloudflare URL; stop it with the following cell when finished.

In [2]:
from getpass import getpass
import re
import time

nim_api_key = os.environ.get("NIM_API_KEY", "").strip()
if not nim_api_key:
    nim_api_key = getpass("NVIDIA NIM API key: ").strip()
if not nim_api_key.startswith("nvapi-"):
    raise ValueError("The supplied NIM_API_KEY is not a valid NVIDIA API key")
os.environ["NIM_API_KEY"] = nim_api_key

checkpoint = SHARED_ROOT / "checkpoints" / "best.pt"
databases = list((SHARED_ROOT / "embeddings").glob("*/lancedb"))
if not checkpoint.is_file():
    raise FileNotFoundError(f"Detector checkpoint not found: {checkpoint}")
if not databases:
    raise FileNotFoundError("No persisted LanceDB artifact was found")

cloudflared = Path("/content/cloudflared")
cloudflared_download = Path("/content/cloudflared.download")
cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/"
    "download/cloudflared-linux-amd64"
)

def cloudflared_is_valid(path):
    if not path.is_file():
        return False
    try:
        result = subprocess.run(
            [str(path), "--version"],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=15,
        )
    except (OSError, subprocess.SubprocessError):
        return False
    return "cloudflared version" in result.stdout.lower()

if not cloudflared_is_valid(cloudflared):
    cloudflared.unlink(missing_ok=True)
    cloudflared_download.unlink(missing_ok=True)
    print("Downloading cloudflared (automatic retries enabled)...")
    subprocess.run(
        [
            "curl",
            "--fail",
            "--location",
            "--retry",
            "5",
            "--retry-all-errors",
            "--connect-timeout",
            "30",
            "--output",
            str(cloudflared_download),
            cloudflared_url,
        ],
        check=True,
    )
    if cloudflared_download.stat().st_size < 10_000_000:
        raise RuntimeError("Downloaded cloudflared file is unexpectedly small")
    cloudflared_download.chmod(0o755)
    if not cloudflared_is_valid(cloudflared_download):
        raise RuntimeError("Downloaded cloudflared executable failed validation")
    cloudflared_download.replace(cloudflared)

for process_name in ("tunnel_process", "streamlit_process"):
    previous = globals().get(process_name)
    if previous is not None and previous.poll() is None:
        previous.terminate()

streamlit_log_path = Path("/content/streamlit.log")
tunnel_log_path = Path("/content/cloudflared.log")
streamlit_log_handle = streamlit_log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        "app/annotation_assistant.py",
        "--server.headless=true",
        "--server.address=127.0.0.1",
        "--server.port=8501",
        "--", "--config", "configs/annotation/single_image.yaml",
    ],
    cwd=REPO_DIR,
    env=os.environ.copy(),
    stdout=streamlit_log_handle,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
if streamlit_process.poll() is not None:
    raise RuntimeError(streamlit_log_path.read_text(errors="replace"))

tunnel_log_handle = tunnel_log_path.open("w")
tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(1)
    tunnel_output = tunnel_log_path.read_text(errors="replace")
    match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", tunnel_output)
    if match:
        public_url = match.group(0)
        break

if public_url is None:
    raise RuntimeError(tunnel_log_path.read_text(errors="replace"))
print(f"Open the annotation assistant: {public_url}")

Open the annotation assistant: https://hampton-count-occurring-deborah.trycloudflare.com


In [3]:
for process_name in ("tunnel_process", "streamlit_process"):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
for handle_name in ("tunnel_log_handle", "streamlit_log_handle"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()
print("Annotation assistant stopped.")

Annotation assistant stopped.
